<a href="https://colab.research.google.com/github/braim/nids-tta/blob/main/colab/5-kan-wav-ae/NIDS_5_3_1_KAN_MMD_WAV_AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code implements a continual learning framework for network intrusion detection using a KAN (Kolmogorov-Arnold Network) autoencoder. It begins by loading and preprocessing multiple network flow datasets (UNSW-NB15, ToN-IoT, CICIDS2018) using Polars, scaling the features, and splitting them into benign training and mixed test sets. A KAN autoencoder is then pre-trained on benign samples from a source dataset. Subsequently, the model undergoes continual adaptation on benign samples from target datasets, leveraging a replay buffer to mitigate catastrophic forgetting. The system evaluates anomaly detection performance by calculating reconstruction error and determining the optimal F1 score and accuracy on mixed test sets throughout the pre-training and adaptation phases.



1.  **`sample_n` (in `load_dataset`)**: This parameter controls the number of samples loaded from each dataset. It directly influences the size of the training and testing data, impacting training time and potentially the model's ability to learn from the full dataset if set too low.
2.  **`scaler` (in `load_dataset`)**: The choice of scaler, specifically `MinMaxScaler(feature_range=(-1, 1))`, is crucial for preprocessing input features. Scaling data to a specific range (e.g., -1 to 1) is often vital for KANs to perform optimally.
3.  **`input_dim` (in `KanAE`)**: This defines the number of features in the input data, which directly determines the first layer's size in the KAN autoencoder architecture. It must match the feature count of the preprocessed datasets.
4.  **`latent_dim` (in `KanAE`)**: This sets the size of the bottleneck layer in the KAN autoencoder. It represents the compressed representation of the input data and is a critical architectural hyperparameter, balancing compression and reconstruction quality.
5.  **`ReplayBuffer(capacity)`**: The `capacity` of the replay buffer dictates how many past benign samples the model can store and replay during continual adaptation. A larger capacity helps in mitigating catastrophic forgetting by providing a broader 'memory' of previous tasks.
6.  **`epochs` (in `train_continual`)**: This parameter specifies the number of training iterations over the dataset for each training phase (pre-training and adaptation). It directly influences how much the model learns and converges, with too few epochs leading to underfitting and too many potentially to overfitting or unnecessary computation.

In [ ]:
# This cell installs necessary packages and is typically run once to set up the environment.
!pip install git+https://github.com/Blealtan/efficient-kan.git
!pip install polars

  Cloning https://github.com/Blealtan/efficient-kan.git to /tmp/pip-req-build-gsdt14t6
  Running command git clone --filter=blob:none --quiet https://github.com/Blealtan/efficient-kan.git /tmp/pip-req-build-gsdt14t6
  Resolved https://github.com/Blealtan/efficient-kan.git to commit 7b6ce1c87f18c8bc90c208f6b494042344216b11
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# Install necessary libraries and import
import importlib.util
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import gc # Garbage Collection


# --- Auto-Install Dependencies ---
packages = {
    'efficient_kan': "git+https://github.com/Blealtan/efficient-kan.git",
    'polars': "polars",
    'kagglehub': "kagglehub"
}

for lib, cmd in packages.items():
    if importlib.util.find_spec(lib) is None:
        print(f"Installing {lib}...")
        !pip install {cmd}

import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve
from efficient_kan import KAN
import kagglehub
import os
import random

In [ ]:
from dataclasses import dataclass

@dataclass
class Hyperparameters:
    # ── Reproducibility ───────────────────────────────────────────────────────────
    seed: int = 42
    # Dataset Params
    sample_n: int = 100000
    input_dim: int = 49

    # Model Params
    latent_dim: int = 8

    # Training Params
    batch_size: int = 128
    pretrain_epochs: int = 20
    adapt_epochs_tgt1: int = 5
    adapt_epochs_tgt2: int = 1
    mmd_weight_tgt1: float = 0.5

    # Buffer Params
    buffer_capacity: int = 10000

config = Hyperparameters()
print(f"Hyperparameters: {config}")

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
np.random.seed(config.seed)
random.seed(config.seed)
torch.backends.cudnn.deterministic = True

In [ ]:


# ==========================================
# 1. POLARS DATA LOADING & PREPROCESSING (Optimized)
# ==========================================

# --- 1. THE FEATURE ENGINEER (Polars Version) ---
def engineer_features(df_pl):
    """
    Accepts a POLARS DataFrame.
    Performs all engineering in-engine (lazy/eager) for max speed and low RAM.
    """
    # 1. Duration
    if 'FLOW_END_MILLISECONDS' in df_pl.columns and 'FLOW_START_MILLISECONDS' in df_pl.columns:
        df_pl = df_pl.with_columns(
            (pl.col('FLOW_END_MILLISECONDS') - pl.col('FLOW_START_MILLISECONDS')).alias('FLOW_DURATION')
        )
    else:
        df_pl = df_pl.with_columns(pl.lit(0).alias('FLOW_DURATION'))

    # 2. Ratios
    if 'IN_BYTES' in df_pl.columns and 'IN_PKTS' in df_pl.columns:
        df_pl = df_pl.with_columns(
            (pl.col('IN_BYTES') / (pl.col('IN_PKTS') + 1e-5)).alias('BYTES_PER_PKT')
        )

    # 3. Log Transform
    cols_to_log = [
        'IN_BYTES', 'IN_PKTS', 'FLOW_DURATION',
        'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV',
        'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_STDDEV',
        'DST_TO_DST_IAT_MAX' # Added in case of typo variations
    ]
    # Filter for existing columns
    existing_log = [c for c in cols_to_log if c in df_pl.columns]

    if existing_log:
        df_pl = df_pl.with_columns(
            [pl.col(c).log1p() for c in existing_log]
        )

    # 4. Drop Artifacts & Targets
    drop_cols = [
        'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS',
        'IPV4_SRC_ADDR', 'IPV4_DST_ADDR',
        'L4_SRC_PORT', 'L4_DST_PORT',
        'Unnamed: 0', 'Dataset', 'Sample_Type',
        'Label', 'Attack', 'label', 'attack',
        'Date', 'IPV6_SRC_ADDR', 'IPV6_DST_ADDR'
    ]
    existing_drop = [c for c in drop_cols if c in df_pl.columns]
    df_pl = df_pl.drop(existing_drop)

    return df_pl

# --- 2. THE LOADER (Pure Polars -> Numpy) ---
def load_dataset(dataset_name, sample_n=100000, scaler=None):
    """
    Loads NF-v3 dataset entirely in Polars, engineers features, and returns Numpy.
    Skips Pandas entirely for memory efficiency.
    """
    print(f"⚡ Loading {dataset_name}...")

    try:
        # 1. Find CSV
        download_path = kagglehub.dataset_download(dataset_name)
        csv_files = [os.path.join(r, f) for r, d, files in os.walk(download_path) for f in files if f.endswith('.csv')]
        if not csv_files: raise FileNotFoundError("No CSV found")
        filepath = csv_files[0]

        # 2. Scan & Collect (Polars)
        q = pl.scan_csv(filepath)
        df_pl = q.collect()

        if sample_n and df_pl.height > sample_n:
            df_pl = df_pl.sample(n=sample_n, seed=42)

        # 3. Extract Label (y)
        label_col = next((c for c in df_pl.columns if c.lower() == 'label'), None)
        if label_col:
            y = df_pl[label_col].to_numpy()
        else:
            y = np.zeros(df_pl.height)

        # 4. Feature Engineering (Pure Polars)
        # This keeps everything efficient.
        df_pl = engineer_features(df_pl)

        # 5. Zero-Copy conversion to Numpy (if possible) or direct cast
        X = df_pl.to_numpy()

        # Cleanup Polars object immediately
        del df_pl
        gc.collect()

        # 6. Final Sanitization
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        X = X.astype(np.float32)

        print(f"   -> Shape: {X.shape} | Attack Rate: {np.mean(y):.2%}")
        return X, y, scaler

    except Exception as e:
        print(f"Error loading {dataset_name}: {e}")
        return None, None, None



def mmd_loss(source_samples, target_samples, sigma=1.0):
    """
    Computes the Maximum Mean Discrepancy (MMD) between two sets of samples.
    Uses a Gaussian kernel.
    """
    def gaussian_kernel(x, y, sigma):
        dist = torch.cdist(x, y, p=2)**2
        return torch.exp(-dist / (2 * sigma**2))

    xx = gaussian_kernel(source_samples, source_samples, sigma)
    yy = gaussian_kernel(target_samples, target_samples, sigma)
    xy = gaussian_kernel(source_samples, target_samples, sigma)

    return xx.mean() + yy.mean() - 2 * xy.mean()
# ==========================================
# MODEL: KAN-AUTOENCODER
# ==========================================
class KanAE(nn.Module):
    def __init__(self, input_dim=49, latent_dim=8):
        super(KanAE, self).__init__()
        # Encoder
        self.encoder = KAN([input_dim, 32, 16, latent_dim], grid_range=[-1, 1])
        self.ln = nn.LayerNorm(latent_dim)
        # Decoder
        self.decoder = KAN([latent_dim, 16, 32, input_dim], grid_range=[-1, 1])

    def forward(self, x):
        z = self.encoder(x)
        z = self.ln(z)  # Apply Layer Norm to latent
        x_recon = self.decoder(z)
        return x_recon, z

print("MMD Loss defined and KanAE class updated with LayerNorm.")




# ==========================================
# MODEL: WAVKAN-AUTOENCODER
# ==========================================
class WavKANLinear(nn.Module):
    def __init__(self, in_features, out_features, wavelet_type='mexican_hat'):
        super(WavKANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.wavelet_type = wavelet_type
        self.translation = nn.Parameter(torch.zeros(out_features, in_features))
        self.scale = nn.Parameter(torch.ones(out_features, in_features))
        self.weights = nn.Parameter(torch.Tensor(out_features, in_features))
        self.base_weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5))
        nn.init.normal_(self.weights, mean=0.0, std=0.01)
        nn.init.uniform_(self.translation, -1.0, 1.0)
        nn.init.uniform_(self.scale, 1.5, 3.5)

    def _mexican_hat(self, x):
        return (1 - x**2) * torch.exp(-0.5 * x**2)

    def _morlet(self, x):
        return torch.cos(5 * x) * torch.exp(-0.5 * x**2)

    def forward(self, x):
        base_output = F.linear(x, self.base_weight)
        x_expanded = x.unsqueeze(1).expand(-1, self.out_features, -1)
        z = (x_expanded - self.translation) / (torch.abs(self.scale) + 1e-5)
        if self.wavelet_type == 'mexican_hat':
            wavelet_val = self._mexican_hat(z)
        elif self.wavelet_type == 'morlet':
            wavelet_val = self._morlet(z)
        else:
            raise ValueError("Unsupported wavelet type")
        wavelet_output = (wavelet_val * self.weights).sum(dim=2)
        return base_output + wavelet_output

class WavKANAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=10):
        super(WavKANAutoEncoder, self).__init__()
        self.encoder_1 = WavKANLinear(input_dim, 32, wavelet_type='mexican_hat')
        self.encoder_2 = WavKANLinear(32, latent_dim, wavelet_type='mexican_hat')
        self.decoder_1 = WavKANLinear(latent_dim, 32, wavelet_type='mexican_hat')
        self.decoder_2 = WavKANLinear(32, input_dim, wavelet_type='mexican_hat')

    def forward(self, x):
        z = self.encoder_1(x)
        z = F.instance_norm(z.unsqueeze(1)).squeeze(1)
        z = self.encoder_2(z)
        x_recon = self.decoder_1(z)
        x_recon = F.layer_norm(x_recon, x_recon.shape[1:])
        x_recon = self.decoder_2(x_recon)
        return x_recon, z


# ==========================================
# 3. REPLAY BUFFER (For Continual Learning)
# ==========================================
class ReplayBuffer:
    def __init__(self, capacity=1000):
        self.buffer = []
        self.capacity = capacity

    def add(self, batch_tensor):
        for i in range(batch_tensor.shape[0]):
            if len(self.buffer) < self.capacity:
                self.buffer.append(batch_tensor[i].detach().cpu())
            else:
                idx = np.random.randint(0, self.capacity)
                self.buffer[idx] = batch_tensor[i].detach().cpu()

    def sample(self, batch_size):
        if len(self.buffer) < batch_size:
            return torch.stack(self.buffer)
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        return torch.stack([self.buffer[i] for i in indices])

# ==========================================
# 4. TRAINING & EVALUATION UTILS
# ==========================================
def evaluate_anomaly_detection(model, loader, device='cuda', desc="Test", threshold=None):
    model.eval()
    mse_loss = nn.MSELoss(reduction='none')
    all_errors = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            x_recon, _ = model(x)
            loss = torch.mean(mse_loss(x_recon, x), dim=1)
            all_errors.extend(loss.cpu().numpy())
            all_labels.extend(y.numpy())

    errors = np.array(all_errors)
    labels = np.array(all_labels)
    if np.isnan(errors).any():
        errors = np.nan_to_num(errors)

    if threshold is None:
        precisions, recalls, thresholds = precision_recall_curve(labels, errors)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
        best_idx = np.argmax(f1_scores)
        best_f1 = f1_scores[best_idx]
        best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]
        if len(thresholds) == 0: best_thresh = 0.0
        print(f"   [{desc} - SEARCH] Found Best Thresh: {best_thresh:.4f} (F1: {best_f1:.4f})")
        return best_thresh
    else:
        preds = (errors >= threshold).astype(int)
        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds)
        print(f"   [{desc} - FIXED]  Thresh: {threshold:.4f} | F1: {f1:.4f} | Acc: {acc:.2%}")
        return acc, f1

def calibrate_threshold(model, loader_benign, device='cuda', percentile=99.9):
    model.eval()
    mse_loss = nn.MSELoss(reduction='none')
    all_errors = []
    with torch.no_grad():
        for x, _ in loader_benign:
            x = x.to(device)
            x_recon, _ = model(x)
            loss = torch.mean(mse_loss(x_recon, x), dim=1)
            all_errors.extend(loss.cpu().numpy())
    errors = np.array(all_errors)
    new_threshold = np.percentile(errors, percentile)
    max_val = np.max(errors)
    print(f"   [Calibration] Max: {max_val:.4f} | {percentile}th: {new_threshold:.4f} | Thresh Set.")
    return new_threshold

def train_continual(model, loader, buffer, epochs=1, device='cuda', mode='source', mmd_weight=0.5):
    # Update initial LR for source to be higher
    current_lr = 1e-2 if mode == 'source' else 5e-3

    optimizer = torch.optim.AdamW(model.parameters(), lr=current_lr, weight_decay=1e-4)

    # Add Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    criterion = nn.MSELoss()
    model.train()
    model.to(device)

    print(f"   [Training] Mode: {mode} | Epochs: {epochs} | Device: {device}")

    for epoch in range(epochs):
        total_loss = 0
        batch_count = 0
        for x, _ in loader:
            x = x.to(device)

            optimizer.zero_grad()

            if mode == 'source':
                # --- Source Phase: Pre-training (MSE Only) ---
                # Add noise for robustness
                noise = torch.randn_like(x) * 0.01
                x_input = x + noise

                # Forward
                x_recon, z = model(x_input)

                # Loss
                loss = criterion(x_recon, x)

                # Add clean samples to buffer
                buffer.add(x)

            elif mode == 'adapt':
                # --- Adapt Phase: Continual Learning (MSE + MMD) ---
                # 1. Forward pass on CURRENT task data (Target)
                x_recon, z_target = model(x)
                mse_loss = criterion(x_recon, x)

                # 2. Sample from Buffer (Source Memory)
                if len(buffer.buffer) > 100:
                    x_replay = buffer.sample(batch_size=x.size(0)).to(device)
                    # We only need the latent Z from the source samples for MMD
                    _, z_source = model(x_replay)

                    # 3. Calculate MMD Loss
                    # detach z_source? Usually MMD aligns current z to target z distribution.
                    # We want z_target to match z_source distribution.
                    mmd = mmd_loss(z_source, z_target)

                    loss = mse_loss + mmd_weight * mmd
                else:
                    # Fallback if buffer is empty (shouldn't happen if pretrained)
                    loss = mse_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            batch_count += 1

        # Step the scheduler at the end of the epoch
        scheduler.step()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            avg_loss = total_loss / max(1, batch_count)
            current_lr_val = scheduler.get_last_lr()[0]
            print(f"   Epoch {epoch+1}/{epochs} Loss: {avg_loss:.4f} | LR: {current_lr_val:.6f}")

def create_split_loaders(X, y, batch_size=128, scaler=None):
    # Optimized Split: avoid full copy if possible
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    mask_benign = (y_train == 0)
    X_adapt = X_train[mask_benign]
    y_adapt = y_train[mask_benign]

    # Free X_train memory immediately
    del X_train, y_train
    gc.collect()

    if scaler is None:
        scaler = RobustScaler()
        X_adapt = scaler.fit_transform(X_adapt)
        print("   [Scaler] Fitted new scaler on Source Benign data.")
    else:
        X_adapt = scaler.transform(X_adapt)
        print("   [Scaler] Used existing scaler to transform Target data.")

    X_test = scaler.transform(X_test)
    X_adapt = np.clip(X_adapt, -5.0, 5.0)
    X_test = np.clip(X_test, -5.0, 5.0)

    loader_adapt = DataLoader(
        TensorDataset(torch.tensor(X_adapt, dtype=torch.float32), torch.tensor(y_adapt, dtype=torch.float32)),
        batch_size=batch_size,
        shuffle=True
    )
    # Free X_adapt memory
    del X_adapt, y_adapt
    gc.collect()

    loader_test = DataLoader(
        TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)),
        batch_size=batch_size,
        shuffle=False
    )
    # Free X_test memory
    del X_test, y_test
    gc.collect()

    print(f"   -> Loaders created. Raw arrays filtered and deleted.")
    return loader_adapt, loader_test, scaler

MMD Loss defined and KanAE class updated with LayerNorm.


In [ ]:

# ==========================================
# 5. INITIALIZATION & DATA LOADING (Memory Optimized)
# ==========================================
if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using {device}. Starting KAN-IDS Experiment (Full Datasets)...\n")

# Create Loaders - SEQUENTIAL LOADING to save RAM
batch_size = config.batch_size

# 1. LOAD SOURCE (ToN-IoT)
print("\n[1/3] Processing Source: ToN-IoT...")
X_src, y_src, _ = load_dataset('seyhed/nf-ton-iot-v3', sample_n=config.sample_n)
loader_src_train, loader_src_test, scaler = create_split_loaders(X_src, y_src, batch_size, scaler=None)

# Clean up Raw Source Data immediately
del X_src, y_src
gc.collect()
print("   -> Source loaded, packed, and raw data deleted.")


# 2. LOAD TARGET 1 (UNSW-NB15)
print("\n[2/3] Processing Target 1: UNSW-NB15...")
X_tgt1, y_tgt1, _ = load_dataset('seyhed/nf-unsw-nb15-v3', sample_n=config.sample_n, scaler=None) # Scaler handled in create_split_loaders
loader_tgt1_adapt, loader_tgt1_test, _ = create_split_loaders(X_tgt1, y_tgt1, batch_size, scaler=scaler)

# Clean up Raw Target 1 Data
del X_tgt1, y_tgt1
gc.collect()
print("   -> Target 1 loaded, packed, and raw data deleted.")


# 3. LOAD TARGET 2 (CICIDS)
print("\n[3/3] Processing Target 2: CICIDS2018...")
X_tgt2, y_tgt2, _ = load_dataset('seyhed/nf-cicids2018-v3', sample_n=config.sample_n, scaler=None)
loader_tgt2_adapt, loader_tgt2_test, _ = create_split_loaders(X_tgt2, y_tgt2, batch_size, scaler=scaler)

# Clean up Raw Target 2 Data
del X_tgt2, y_tgt2
gc.collect()
print("   -> Target 2 loaded, packed, and raw data deleted.")

print(f"\nData Split Summary (70/30 Split Applied):")
print(f"  Src Train (Benign): {len(loader_src_train.dataset)} | Src Test (Mixed): {len(loader_src_test.dataset)}")
print(f"  Tgt1 Adapt (Benign): {len(loader_tgt1_adapt.dataset)} | Tgt1 Test (Mixed): {len(loader_tgt1_test.dataset)}")
print(f"  Tgt2 Adapt (Benign): {len(loader_tgt2_adapt.dataset)} | Tgt2 Test (Mixed): {len(loader_tgt2_test.dataset)}")

Using cpu. Starting KAN-IDS Experiment (Full Datasets)...


[1/3] Processing Source: ToN-IoT...
⚡ Loading seyhed/nf-ton-iot-v3...
Using Colab cache for faster access to the 'nf-ton-iot-v3' dataset.
   -> Shape: (100000, 49) | Attack Rate: 38.94%
   [Scaler] Fitted new scaler on Source Benign data.
   -> Loaders created. Raw arrays filtered and deleted.
   -> Source loaded, packed, and raw data deleted.

[2/3] Processing Target 1: UNSW-NB15...
⚡ Loading seyhed/nf-unsw-nb15-v3...
   -> Shape: (100000, 49) | Attack Rate: 5.52%
   [Scaler] Used existing scaler to transform Target data.
   -> Loaders created. Raw arrays filtered and deleted.
   -> Target 1 loaded, packed, and raw data deleted.

[3/3] Processing Target 2: CICIDS2018...
⚡ Loading seyhed/nf-cicids2018-v3...
Using Colab cache for faster access to the 'nf-cicids2018-v3' dataset.
   -> Shape: (100000, 49) | Attack Rate: 12.95%
   [Scaler] Used existing scaler to transform Target data.
   -> Loaders created. Raw arrays filtered an

In [ ]:

# Re-initialize Model with requested architecture (Latent Dim = 8)
model = KanAE(input_dim=config.input_dim, latent_dim=config.latent_dim)

# Re-initialize Replay Buffer (Empty)
memory = ReplayBuffer(capacity=config.buffer_capacity)

print(f"\n[System] New KanAE Model Initialized: [Input, 32, 16, {config.latent_dim}]")
print(f"[System] Replay Buffer Reset (Capacity: {config.buffer_capacity})")


print("\n" + "="*40)
print("PHASE 1: PRE-TRAINING (SOURCE: ToN-IoT - Benign Only)")
print("="*40)
# Train on loader_src_train (Benign)
# Reduced epochs to 20 for demonstration, can be increased
train_continual(model, loader_src_train, memory, epochs=config.pretrain_epochs, device=device, mode='source')



[System] New KanAE Model Initialized: [Input, 32, 16, 8]
[System] Replay Buffer Reset (Capacity: 10000)

PHASE 1: PRE-TRAINING (SOURCE: ToN-IoT - Benign Only)
   [Training] Mode: source | Epochs: 20 | Device: cpu
   Epoch 1/20 Loss: 0.1777 | LR: 0.009939
   Epoch 5/20 Loss: 0.0248 | LR: 0.008537
   Epoch 10/20 Loss: 0.0151 | LR: 0.005005
   Epoch 15/20 Loss: 0.0096 | LR: 0.001473
   Epoch 20/20 Loss: 0.0077 | LR: 0.000010


In [ ]:
print("\n" + "="*40)
print("PHASE 2: CROSS-EVALUATION (BASELINE)")
print("="*40)

# 1. Determine Threshold using SOURCE Test (or a dedicated Validation set if we had one)
# We use the source test set to calibrate 'What is an error?'
best_thresh = evaluate_anomaly_detection(model, loader_src_test, device, desc="Source (ToN-IoT) [Calibration]", threshold=None)
print(f"\n[System] Global Anomaly Threshold Set to: {best_thresh:.4f}\n")

# 2. Apply this FIXED threshold to everything else
evaluate_anomaly_detection(model, loader_src_test, device, desc="Source (ToN-IoT)", threshold=best_thresh)
evaluate_anomaly_detection(model, loader_tgt1_test, device, desc="Target 1 (UNSW) [Zero-Shot]", threshold=best_thresh)
evaluate_anomaly_detection(model, loader_tgt2_test, device, desc="Target 2 (CICIDS) [Zero-Shot]", threshold=best_thresh)


PHASE 2: CROSS-EVALUATION (BASELINE)
   [Source (ToN-IoT) [Calibration] - SEARCH] Found Best Thresh: 0.0027 (F1: 0.8270)

[System] Global Anomaly Threshold Set to: 0.0027

   [Source (ToN-IoT) - FIXED]  Thresh: 0.0027 | F1: 0.8270 | Acc: 86.51%
   [Target 1 (UNSW) [Zero-Shot] - FIXED]  Thresh: 0.0027 | F1: 0.1047 | Acc: 5.52%
   [Target 2 (CICIDS) [Zero-Shot] - FIXED]  Thresh: 0.0027 | F1: 0.1894 | Acc: 14.04%


(0.14043333333333333, 0.18941941973407098)

In [ ]:
print("\n" + "="*40)
print("PHASE 3: CONTINUAL ADAPTATION (TARGET 1: UNSW) - MMD")
print("="*40)

# 1. Adapt to UNSW Benign samples while replaying ToN-IoT Benign via MMD
# Using epochs=5 and mmd_weight=0.5 as requested
train_continual(model, loader_tgt1_adapt, memory, epochs=config.adapt_epochs_tgt1, device=device, mode='adapt', mmd_weight=config.mmd_weight_tgt1)

# 2. RECALIBRATE THRESHOLD (Using 95th Percentile)
new_thresh_tgt1 = calibrate_threshold(model, loader_tgt1_adapt, device, percentile=95)

print("\n[POST-ADAPTATION RESULTS]")
# 3. Evaluate on Target
evaluate_anomaly_detection(model, loader_tgt1_test, device, desc="Target 1 (UNSW) [Adapted]", threshold=new_thresh_tgt1)
# 4. Evaluate Retention on Source
evaluate_anomaly_detection(model, loader_src_test, device, desc="Source (ToN-IoT) [Retention Check]", threshold=best_thresh)



PHASE 3: CONTINUAL ADAPTATION (TARGET 1: UNSW) - MMD
   [Training] Mode: adapt | Epochs: 5 | Device: cpu
   Epoch 1/5 Loss: 0.1804 | LR: 0.004523
   Epoch 5/5 Loss: 0.0219 | LR: 0.000010
   [Calibration] Max: 3.5614 | 95th: 0.0505 | Thresh Set.

[POST-ADAPTATION RESULTS]
   [Target 1 (UNSW) [Adapted] - FIXED]  Thresh: 0.0505 | F1: 0.6877 | Acc: 95.02%
   [Source (ToN-IoT) [Retention Check] - FIXED]  Thresh: 0.0027 | F1: 0.5605 | Acc: 38.94%


(0.3893666666666667, 0.5604951896547588)

In [ ]:
print("\n" + "="*40)
print("PHASE 4: CONTINUAL ADAPTATION (TARGET 2: CICIDS)")
print("="*40)

# 1. Adapt to CICIDS Benign
# Note: 'epochs=5' is a moderate adaptation phase. Consider increasing/decreasing for better adaptation.
train_continual(model, loader_tgt2_adapt, memory, epochs=config.adapt_epochs_tgt2, device=device, mode='adapt')

# 2. RECALIBRATE THRESHOLD (Using 85th Percentile)
# Calibrating with 85th percentile as a deliberate choice for optimal results.
new_thresh_tgt2 = calibrate_threshold(model, loader_tgt2_adapt, device, percentile=85)

print("\n[POST-ADAPTATION RESULTS]")
evaluate_anomaly_detection(model, loader_tgt2_test, device, desc="Target 2 (CICIDS) [Adapted]", threshold=new_thresh_tgt2)
evaluate_anomaly_detection(model, loader_src_test, device, desc="Source (ToN-IoT) [Retention Check]", threshold=best_thresh)



PHASE 4: CONTINUAL ADAPTATION (TARGET 2: CICIDS)
   [Training] Mode: adapt | Epochs: 1 | Device: cpu
   Epoch 1/1 Loss: 0.2937 | LR: 0.000010
   [Calibration] Max: 9.8884 | 85th: 0.2121 | Thresh Set.

[POST-ADAPTATION RESULTS]
   [Target 2 (CICIDS) [Adapted] - FIXED]  Thresh: 0.2121 | F1: 0.6449 | Acc: 86.42%
   [Source (ToN-IoT) [Retention Check] - FIXED]  Thresh: 0.0027 | F1: 0.5605 | Acc: 38.94%


(0.3893666666666667, 0.5604951896547588)